# 🚀 Training Dual-Stream YOLOv11 di Kaggle dengan Dataset Primer (`hujan-2`)

Notebook ini melatih model **Dual-Stream YOLOv11** dengan struktur dataset primer:
```text
/kaggle/input/hujan-2/mydata/
├── train/
│   ├── images/ (ir/, vi/)
│   └── labels/ (vi/)
└── val/
    ├── images/ (ir/, vi/)
    └── labels/ (vi/)
```

### 1. Cek GPU & Install Dependensi

In [ ]:
# Cek GPU
!nvidia-smi

# Install Ultralytics
!pip install -q ultralytics

### 2. Clone Repositori Dual-Stream YOLOv11

In [ ]:
import os

%cd /kaggle/working

if not os.path.exists('Dual-Stream-YOLOv11'):
    !git clone https://github.com/adityanhh/Dual-Stream-YOLOv11.git

%cd /kaggle/working/Dual-Stream-YOLOv11
print("✅ Masuk ke direktori Dual-Stream-YOLOv11")

### 3. Verifikasi Jumlah Citra & Label Dataset `hujan-2`

In [ ]:
import glob

base_path = "/kaggle/input/hujan-2/mydata"

train_vi = glob.glob(f"{base_path}/train/images/vi/*.*")
train_ir = glob.glob(f"{base_path}/train/images/ir/*.*")
train_lbl = glob.glob(f"{base_path}/train/labels/vi/*.txt")

val_vi = glob.glob(f"{base_path}/val/images/vi/*.*")
val_ir = glob.glob(f"{base_path}/val/images/ir/*.*")
val_lbl = glob.glob(f"{base_path}/val/labels/vi/*.txt")

print(f"📊 Ringkasan Dataset Hujan-2:")
print(f" - Train VI (Visible)  : {len(train_vi)} file")
print(f" - Train IR (Thermal)  : {len(train_ir)} file")
print(f" - Train Labels        : {len(train_lbl)} file")
print(f" - Val VI (Visible)    : {len(val_vi)} file")
print(f" - Val IR (Thermal)    : {len(val_ir)} file")
print(f" - Val Labels          : {len(val_lbl)} file")

### 4. Download Bobot Pretrained YOLO11 (Transfer Learning)

In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt
print("✅ Bobot yolo11n.pt siap digunakan untuk transfer learning!")

### 5. Mulai Pelatihan Model (Training)

In [ ]:
!python train.py \
    --model configs/yolo11n-dualstream.yaml \
    --data configs/hujan2.yaml \
    --pretrained yolo11n.pt \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --device 0 \
    --project /kaggle/working/runs/train \
    --name exp_hujan2

### 6. Uji Inferensi (Prediction) pada Pasangan Citra Validasi

In [ ]:
if val_vi and val_ir:
    sample_vis = val_vi[0]
    sample_ir = val_ir[0]
    
    !python predict.py \
        --weights /kaggle/working/runs/train/exp_hujan2/best.pt \
        --model configs/yolo11n-dualstream.yaml \
        --vis-img "{sample_vis}" \
        --ir-img "{sample_ir}" \
        --conf 0.25 \
        --save-dir /kaggle/working/runs/predict

### 7. Kompres & Download Hasil Pelatihan (`best.pt` & Checkpoints)

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/hujan2_training_results', 'zip', '/kaggle/working/runs/train/exp_hujan2')
print("🎉 Selesai! File 'hujan2_training_results.zip' siap di-download langsung dari panel Output!")